# Proyek 2: Klasifikasi Sentimen Multi-Kelas pada Teks Komentar Bahasa Indonesia

## Notebook 1: Preprocessing & EDA

## 1. Identifikasi Proyek & Sumber Data
* **Nama Proyek:** Multi-class Sentiment Analysis untuk Teks Media Sosial Bahasa Indonesia
* **Sumber Dataset:** Dataset Publik [Alvian di Kaggle (Indonesian Sentiment Analysis - IndoNLU)](https://www.kaggle.com/datasets/alvianhanafie/indonesian-sentiment-analysis-indonlu)
* **Koleksi Korpus:** Korpus *SmSA* (Sms Sentiment Analysis / Ulasan Teks) dari repositori populer **IndoNLU**
* **Direktori Penyimpanan:** Berkas mentah disimpan pada direktori lokal `../data/raw/` dengan format `.tsv` (*Tab-Separated Values*).

## 2. Karakteristik Dataset & Spesifikasi Fitur
Dataset yang disusun oleh Alvian di Kaggle ini bersumber langsung dari data riil opini, komentar, dan ulasan masyarakat Indonesia di ranah digital. Karakteristik utamanya meliputi:
1. **Kekayaan Bahasa Kasual:** Data merekam gaya komunikasi netizen sehari-hari yang sangat dinamis, kaya akan elemen kosakata tidak baku, singkatan (*slang words*), *typo*, perulangan huruf, serta pencampuran dialek lokal.
2. **Struktur Pre-split (Terpartisi):** Dataset ini secara bawaan telah terbagi secara terpisah menjadi 3 partisi file yaitu *Train* (Data Latih), *Validation* (Data Validasi), dan *Test* (Data Uji). Pemisahan sejak awal ini sangat ideal dalam *Data Science* untuk menghindari kebocoran data (*data leakage*) selama eksperimen model pada notebook berikutnya.

### Rincian Struktur Fitur:
* **`text`** (*Fitur Utama / Object*): String berupa untaian kalimat atau dokumen ulasan mentah dari pengguna digital yang akan diekstrak maknanya.
* **`sentiment`** (*Target Label / Kategori*): Variabel target multi-kelas (*multi-class*) yang merepresentasikan orientasi emosi teks, terbagi secara ketat menjadi 3 kelas:
  * `positive` (Opini bersifat mendukung, puas, atau berupa pujian).
  * `neutral` (Opini bersifat objektif, datar, atau sekadar membagikan informasi).
  * `negative` (Opini bersifat kritikan, kekecewaan, atau komplain).

---

## 3. Pendahuluan & Tujuan Proyek
With memahami karakteristik dataset dari Kaggle Alvian di atas, proyek ini bertujuan untuk membangun model komputasi cerdas yang mampu mengenali dan mengklasifikasikan orientasi sentimen secara otomatis menggunakan pendekatan *Machine Learning* dan *Natural Language Processing* (NLP).

**Tujuan Spesifik Notebook 1 (NB1):**
1. **Memahami Karakteristik Data (EDA):** Melakukan audit dimensi baris data, mengeliminasi kontaminasi data duplikat pada subset data, dan mengidentifikasi tingkat ketidakseimbangan kelas (*class imbalance*).
2. **Pembersihan Teks (Preprocessing):** Menyusun fungsi *preprocessing modular* menggunakan ekspresi reguler (regex) untuk menyaring noise teks digital (seperti URL, mention, hashtag, angka, dan tanda baca) serta menormalisasi kosakata kasual agar menghasilkan representasi kata baku yang siap diolah ke tahap pembobotan numerik pada Notebook 2.

---

## 4. Alur & Langkah Pengerjaan
Proses di dalam Notebook 1 ini dirancang secara linear ke dalam 6 tahapan utama:
1. **Preparation & Data Loading:** Memuat partisi dataset dari folder raw dan membersihkan noise baris (data duplikat).
2. **Uji Coba Tahap 1 (Text Cleansing):** Eksperimen *Regular Expression* untuk membuang URL, mention, hashtag, angka, dan tanda baca pada satu kalimat contoh.
3. **Uji Coba Tahap 2 (Word Normalization):** Eksperimen pemecahan kata (*tokenization*) dan konversi kata gaul memakai kamus pemetaan internal.
4. **Eksekusi Pipeline Massal:** Menerapkan fungsi pembersihan terpadu ke seluruh baris dataframe menggunakan metode `.apply()`, dilengkapi audit string kosong setelah proses pembersihan.
5. **Analisis Eksploratif Target (EDA Target):** Menghitung distribusi persentase label target untuk menentukan strategi penanganan data timpang dan metrik evaluasi model.
6. **Data Export:** Mengekspor hasil pembersihan akhir ke folder `../data/processed/` dalam format `.tsv` agar siap digunakan pada Notebook 2.

## LANGKAH 1: PREPARATION & DATA LOADING

Pada tahapan pertama ini, kita membangun sebuah fungsi modular terintegrasi bernama `load_and_audit_data`. Fungsi ini bertugas memuat tiga partisi dataset (*Train*, *Validation*, dan *Test*) dari folder penyimpanan lokal, mendeteksi keberadaan nilai kosong (*missing values*), serta secara otomatis mengeliminasi kontaminasi data duplikat pada subset data latih (*Train*). Hal ini krusial agar model tidak mempelajari informasi bias secara berulang dari kalimat yang sama (*data replication bias*).

In [1]:
# ==============================================================================
# TAHAPAN 1: PREPARATION & DATA LOADING & AUDIT DATA (MODULAR VERSION - FIXED)
# ==============================================================================
import os
import sys
import pandas as pd

def load_and_audit_data(train_path, valid_path, test_path):
    """
    Memuat dataset dari berkas .tsv, melakukan audit kualitas data awal 
    (pengecekan missing values), membersihkan baris data duplikat pada subset Train,
    serta menginisialisasi kamus normalisasi bahasa gaul.

    Parameters:
    ----------
    train_path : str
        Path lokasi penyimpanan file tsv untuk data training (data latih).
    valid_path : str
        Path lokasi penyimpanan file tsv untuk data validation (data validasi).
    test_path : str
        Path lokasi penyimpanan file tsv untuk data test (data uji).

    Returns:
    -------
    tuple
        Mengembalikan tiga pandas.DataFrame dan satu dictionary kamus: 
        (df_train, df_valid, df_test, kamus_map).
    """
    # 1. Memuat dataset menggunakan pandas
    print("=== 1. Memuat Dataset Mentah ===")
    df_train = pd.read_csv(train_path, sep='\t')
    df_valid = pd.read_csv(valid_path, sep='\t')
    df_test = pd.read_csv(test_path, sep='\t')
    
    print(f"Data Train Awal      : {df_train.shape[0]} baris, {df_train.shape[1]} kolom")
    print(f"Data Validation Awal : {df_valid.shape[0]} baris, {df_valid.shape[1]} kolom")
    print(f"Data Test Awal       : {df_test.shape[0]} baris, {df_test.shape[1]} kolom\n")
    
    # 2. Pengecekan Missing Values
    print("=== 2. Pengecekan Missing Values ===")
    print(f"Missing values di kolom 'text' (Train)      : {df_train['text'].isnull().sum()}")
    print(f"Missing values di kolom 'sentiment' (Train) : {df_train['sentiment'].isnull().sum()}\n")
    
    # 3. Audit & Pembersihan Data Duplikat
    print("=== 3. Audit & Pembersihan Data Duplikat ===")
    duplicate_count = df_train.duplicated(subset=['text']).sum()
    print(f"Jumlah data duplikat ditemukan di subset Train: {duplicate_count}")
    
    if duplicate_count > 0:
        df_train = df_train.drop_duplicates(subset=['text']).reset_index(drop=True)
        print(f"--> [SUCCESS] {duplicate_count} data duplikat berhasil dihapus!")
        print(f"--> Jumlah baris Data Train sekarang: {df_train.shape[0]} baris.\n")
        
    # 4. Inisialisasi Kamus Normalisasi Kata Gaul (Ditaruh di dalam fungsi)
    kamus_map = {
        "bgt": "banget", "bgtt": "banget", "gk": "tidak", "gak": "tidak", "ga": "tidak",
        "tdk": "tidak", "jg": "juga", "jgn": "jangan", "klo": "kalau", "kl": "kalau",
        "yg": "yang", "dgn": "dengan", "utk": "untuk", "tp": "tetapi", "tpi": "tetapi",
        "bisaa": "bisa", "bener": "benar", "moga": "semoga", "emang": "memang",
        "udh": "sudah", "udah": "sudah", "deh": "sudah", "top": "bagus", "kerenn": "keren",
        "aja": "saja", "aj": "saja", "aq": "saya", "ak": "saya", "lu": "kamu", "lo": "kamu",
        "ngga": "tidak"
    }
    print(f"--> [SUCCESS] Kamus internal siap dengan {len(kamus_map)} kosakata gaul.\n")
        
    print("=== 5. Menampilkan 3 Sampel Data Teratas (Train) ===")
    print(df_train.head(3))
    print("\n" + "="*50 + "\n--> [SUCCESS] Fungsi audit sukses dieksekusi!\n" + "="*50)
    
    # Mengembalikan 4 output sekaligus ke memori global notebook
    return df_train, df_valid, df_test, kamus_map

# 4. Menentukan path berkas mentah (.tsv)
train_file = "../data/raw/train_preprocess_ori.tsv"
valid_file = "../data/raw/valid_preprocess.tsv"
test_file = "../data/raw/test_preprocess_masked_label.tsv"

# 5. Memanggil fungsi modular untuk mengunci dataframe & kamus ke memori global
df_train, df_valid, df_test, kamus_map = load_and_audit_data(train_file, valid_file, test_file)

=== 1. Memuat Dataset Mentah ===
Data Train Awal      : 11000 baris, 2 kolom
Data Validation Awal : 1260 baris, 2 kolom
Data Test Awal       : 500 baris, 2 kolom

=== 2. Pengecekan Missing Values ===
Missing values di kolom 'text' (Train)      : 0
Missing values di kolom 'sentiment' (Train) : 0

=== 3. Audit & Pembersihan Data Duplikat ===
Jumlah data duplikat ditemukan di subset Train: 67
--> [SUCCESS] 67 data duplikat berhasil dihapus!
--> Jumlah baris Data Train sekarang: 10933 baris.

--> [SUCCESS] Kamus internal siap dengan 31 kosakata gaul.

=== 5. Menampilkan 3 Sampel Data Teratas (Train) ===
                                                text sentiment
0  warung ini dimiliki oleh pengusaha pabrik tahu...  positive
1  mohon ulama lurus dan k212 mmbri hujjah partai...   neutral
2  lokasi strategis di jalan sumatera bandung . t...  positive

--> [SUCCESS] Fungsi audit sukses dieksekusi!


### INSIGHT & ANALISIS HASIL TAHAPAN 1: LOADING & AUDIT DATA

Berdasarkan hasil eksekusi melalui fungsi modular `load_and_audit_data` di atas, diperoleh tiga poin analisis utama yang memvalidasi kualitas data awal kita:

1. **Implementasi Fungsi Modular Berhasil Sempurna**
   Penerapan fungsi tunggal untuk memuat sekaligus mengaudit data terbukti meningkatkan keterbacaan kode (*code readability*) dan efisiensi memori global. Penulisan *docstring* yang terstruktur membantu mendokumentasikan parameter masukan (*inputs*) dan keluaran (*outputs*) secara transparan, sesuai dengan standar industri untuk kebutuhan portofolio.

2. **Integritas Struktur Data dan Validasi Missing Values**
   Dataset dari Kaggle Alvian berhasil dimuat ke memori tanpa adanya kegagalan pembacaan pembatas tabulator (*parsing error*). Pengecekan *missing values* menunjukkan angka **0 (nol)** pada kolom `text` maupun `sentiment` di subset Train. Ini menandakan kualitas kerapian sensor data mentah dari sumbernya sudah sangat baik.

3. **Pembersihan Noise Duplikat dan Rasionalisasi Data Latih**
   Fungsi berhasil mendeteksi dan memangkas **67 baris data duplikat** berbasis teks pada subset *Train*. 
   * **Justifikasi Metodologis:** Pembersihan data duplikat sengaja **hanya difokuskan pada subset Train** (dari 11.000 menyusut menjadi 10.933 baris). Kami membiarkan subset *Validation* (1.260 baris) dan *Test* (500 baris) apa adanya untuk menjaga keaslian sebaran distribusi dokumen di lapangan, sehingga evaluasi model tetap mencerminkan situasi dunia nyata.
   * **Inisialisasi Kamus Slang:** Fungsi ini juga mengunci objek `kamus_map` berisi **31 kosakata gaul** ke memori global, yang nantinya akan dialirkan sebagai komponen utama penormalan teks pada *pipeline preprocessing* massal.

## LANGKAH 2: PEMBERSIHAN TEKS TAHAP AWAL (TEXT CLEANSING & CASE FOLDING)

Sebelum melakukan pemrosesan secara massal pada seluruh baris dataset, kita perlu menguji fungsi dasar pembersihan pada satu sampel kalimat teks. Langkah pertama ini berfokus pada reduksi huruf kapital (*case folding*) serta pembersihan *noise* struktural teks digital menggunakan bantuan *Regular Expression* (Regex). Karakter pengganggu seperti tautan web (URL), sapaan akun (*mention*), penanda topik (*hashtag*), angka, dan tanda baca akan disaring sepenuhnya.

In [2]:
# ==============================================================================
# TAHAPAN 2: UJI COBA TEXT CLEANSING & CASE FOLDING (1 SAMPEL TEKS)
# ==============================================================================
import re

def initial_text_clean(text):
    """
    Melakukan pembersihan teks tahap awal secara komprehensif meliputi case folding,
    penghapusan URL, mention, hashtag, karakter non-alfabet, dan perapian spasi ganda.
    """
    # 1. Case Folding: Menyeragamkan semua huruf menjadi huruf kecil
    text = text.lower()
    
    # 2. Menghapus URL/Tautan internet
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    
    # 3. Menghapus Mention (@username) dan Hashtag (#tagar)
    text = re.sub(r'@\S+|#\S+', '', text)
    
    # 4. Menghapus karakter non-alfabet (angka, tanda baca, simbol, emoji)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    
    # 5. Menghapus whitespace ganda akibat dampak penghapusan karakter
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Menyiapkan satu sampel kalimat teks kasual/media sosial yang kompleks
contoh_teks = "Wow!! Keren bgt produknya @toko_bagus... Belanja di https://tokobagus.com #puas deh pokoknya!! 100% TOP!"

print("Teks Asli Mentah :", contoh_teks)
print("Hasil Text Clean :", initial_text_clean(contoh_teks))

Teks Asli Mentah : Wow!! Keren bgt produknya @toko_bagus... Belanja di https://tokobagus.com #puas deh pokoknya!! 100% TOP!
Hasil Text Clean : wow keren bgt produknya belanja di deh pokoknya top


### INSIGHT & ANALISIS HASIL TAHAPAN 2: TEXT CLEANSING & CASE FOLDING

Dari hasil uji coba fungsi `initial_text_clean` pada satu kalimat sampel media sosial yang kompleks di atas, berikut adalah analisis transformasi string yang terjadi:

1. **Pembersihan *Noise* Digital Berhasil Total**
   Karakter non-informatif berhasil disaring secara bersih tanpa merusak struktur spasial kalimat. Ini meliputi tanda baca ganda (`!!`, `...`), simbol matematika (`%`), angka numerik (`100`), tautan internet (`https://tokobagus.com`), serta entitas media sosial seperti *mention* (`@toko_bagus`) dan *hashtag* (`#puas`).

2. **Standardisasi Token via *Case Folding***
   Kata kapital yang berada di awal kalimat (`Wow`) dan kata penekanan di akhir kalimat (`TOP`) berhasil diubah secara seragam menjadi huruf kecil (`wow` dan `top`). Langkah penyeragaman ini sangat krusial agar algoritma ekstraksi fitur (TF-IDF) tidak menganggap kata kapital dan kata huruf kecil sebagai dua entitas fitur (*token*) yang berbeda.

3. **Perapian Ruang Kosong (*Whitespace Management*)**
   Penggunaan regex `\s+` yang diakhiri metode `.strip()` terbukti efektif mereduksi sisa-sisa spasi ganda yang ditinggalkan oleh karakter yang dihapus, sehingga menghasilkan untaian string baru yang rapi dan rapat.

4. **Justifikasi Kebutuhan Normalisasi Lanjutan**
   Jika diperhatikan pada teks hasil keluaran: `'wow keren bgt produknya belanja di deh pokoknya top'`, kalimat tersebut kini murni hanya berisi karakter alfabet, namun masih menyisakan kosakata tidak baku (*slang words*) seperti `bgt`, `deh`, dan `top`. 
   
   Kondisi ini merupakan hal yang ideal, karena fungsi Langkah 2 ini memang dirancang khusus hanya untuk menyaring karakter pengganggu fisik teks (*cleansing*). Untuk menyamakan persepsi semantik kata-kata gaul tersebut dengan kamus baku Bahasa Indonesia, kita akan mengalirkan teks bersih ini ke tahap berikutnya, yaitu **Normalisasi Kata Menggunakan Kamus Pemetaan**.

## LANGKAH 3: PEMBERSIHAN TEKS TAHAP LANJUTAN (TOKENIZATION & WORD NORMALIZATION)

Setelah teks berhasil dibersihkan dari *noise* komponen non-alfabet pada Langkah 2, tantangan berikutnya adalah menangani kosakata tidak baku (*slang/bahasa gaul*). Pada tahapan ini, kita menguji fungsi untuk memecah string kalimat menjadi potongan kata tunggal (*tokenization*) memanfaatkan spasi murni, lalu mencocokkan setiap token tersebut dengan sebuah kamus pemetaan bahasa gaul (*dictionary lookup*) untuk mentransformasikannya menjadi kosakata baku standar Bahasa Indonesia.

In [3]:
# ==============================================================================
# TAHAPAN 3: UJI COBA TOKENISASI & NORMALISASI KATA GAUL (1 SAMPEL TEKS)
# ==============================================================================

def tokenize_and_normalize(text, dictionary):
    """
    Memecah teks menjadi potongan kata (tokenisasi) sekaligus mengubah
    kata gaul/singkatan menjadi kata baku berdasarkan kamus pemetaan biner.
    """
    # 1. Tokenisasi: Memotong kalimat panjang menjadi list kata per kata menggunakan spasi
    words = text.split()
    
    # 2. Normalisasi: Mengganti kata jika terdaftar di kamus alay, jika tidak ada tetap pakai kata asli
    normalized_words = [dictionary.get(word, word) for word in words]
    
    return normalized_words

# Kita ambil string hasil bersih dari Langkah 2 sebelumnya
teks_bersih_sebelumnya = "wow keren bgt produknya belanja di deh pokoknya top"

print("Teks Bersih Awal :", teks_bersih_sebelumnya)
print("Hasil Tokenisasi & Normalisasi (List):")
print(tokenize_and_normalize(teks_bersih_sebelumnya, kamus_map))

Teks Bersih Awal : wow keren bgt produknya belanja di deh pokoknya top
Hasil Tokenisasi & Normalisasi (List):
['wow', 'keren', 'banget', 'produknya', 'belanja', 'di', 'sudah', 'pokoknya', 'bagus']


### INSIGHT & ANALISIS HASIL TAHAPAN 3: TOKENIZATION & WORD NORMALIZATION

Berdasarkan hasil eksekusi fungsi uji coba penormalan kata di atas, diperoleh tiga poin analisis utama yang memvalidasi efektivitas transformasi kata:

1. **Mekanisme Tokenisasi Berhasil Sempurna**
   Fungsi bawaan `.split()` sukses memotong struktur string kalimat panjang menjadi potongan kata mandiri di dalam struktur data *List* Python (`['wow', 'keren', ...]`). Pemisahan ini merupakan prasyarat wajib dalam *Natural Language Processing* (NLP) agar tiap-tiap token kata siap dialirkan ke tahap pembobotan statistik numerik pada proses *feature engineering*.

2. **Transformasi *Slang Words* ke Kosakata Baku Sangat Akurat**
   Kosakata kasual digital yang tadinya ambigu bagi mesin berhasil dipulihkan secara instan ke bentuk baku standarnya berdasarkan isi objek `kamus_map`:
   * Token `'bgt'` sukses dikonversi menjadi `'banget'`.
   * Token `'deh'` berhasil dinormalisasi menjadi `'sudah'`.
   * Token `'top'` berhasil diterjemahkan secara semantik sebagai `'bagus'`.
   
   Token yang tidak terdaftar di dalam kamus (seperti `'wow'`, `'keren'`, dan `'produknya'`) tetap dipertahankan pada bentuk aslinya tanpa mengalami kerusakan karakter.

3. **Validasi Kesiapan Skala Massal**
   Pengujian ini membuktikan bahwa logika pencarian berbasis *hash table* (`dictionary.get()`) menggunakan *List Comprehension* berjalan dengan sangat efisien dengan kompleksitas waktu yang konstan. Struktur ini dinyatakan **valid dan aman** untuk diintegrasikan ke dalam fungsi *pipeline* besar guna memproses belasan ribu baris dataset secara massal pada tahapan berikutnya.

## LANGKAH 4: EKSEKUSI PIPELINE PEMBERSIHAN TEKS SECARA MASSAL & AUDIT DATA

Setelah memvalidasi keberhasilan fungsi pembersihan dan penormalan kata pada sampel kecil, langkah selanjutnya adalah menyatukannya ke dalam satu fungsi terintegrasi bernama `total_clean_pipeline`. Fungsi ini akan mengeksekusi *preprocessing* secara serentak pada seluruh baris di ketiga partisi dataframe (*Train*, *Validation*, dan *Test*). 

Untuk memastikan integritas data tetap terjaga, kita juga menambahkan proses audit guna mendeteksi dan mengeliminasi baris teks yang berubah menjadi string kosong pasca-pembersihan (misalnya ulasan mentah yang isinya hanya berupa kumpulan emoji, angka, atau simbol).

In [4]:
# ==============================================================================
# TAHAPAN 4: EXECUTE PIPELINE MASSAL & AUDIT STRING KOSONG (OPTIMIZED VERSION)
# ==============================================================================
import numpy as np  # Ditambahkan untuk standardisasi missing values (NaN)

def total_clean_pipeline(text, dictionary):
    """
    Pipeline terintegrasi yang menggabungkan text cleansing awal,
    tokenisasi, normalisasi kata gaul, dan detokenisasi kembali menjadi string kalimat.
    """
    # 1. Jalankan Cleansing awal (Regex)
    text_cleaned = initial_text_clean(text)
    
    # 2. Pecah menjadi list kata
    words = text_cleaned.split()
    
    # 3. Normalisasi kata menggunakan kamus_map
    normalized_words = [dictionary.get(word, word) for word in words]
    
    # 4. Detokenisasi: Menggabungkan kembali list kata menjadi string kalimat utuh
    return " ".join(normalized_words)


print("=== 1. Memulai Proses Preprocessing Massal... ===")
# Menerapkan fungsi ke setiap baris menggunakan lambda dan .apply()
df_train['text_clean'] = df_train['text'].apply(lambda x: total_clean_pipeline(x, kamus_map))
df_valid['text_clean'] = df_valid['text'].apply(lambda x: total_clean_pipeline(x, kamus_map))
df_test['text_clean'] = df_test['text'].apply(lambda x: total_clean_pipeline(x, kamus_map))
print("--> [SUCCESS] Pembuatan kolom 'text_clean' pada seluruh dataset selesai!\n")


print("=== 2. Audit & Penanganan String Kosong ===")
# PERBAIKAN UTAMA: Mengubah string kosong/spasi menjadi np.nan (bukan None) agar aman di Pandas terbaru
df_train['text_clean'] = df_train['text_clean'].replace(r'^\s*$', np.nan, regex=True)
df_valid['text_clean'] = df_valid['text_clean'].replace(r'^\s*$', np.nan, regex=True)
df_test['text_clean'] = df_test['text_clean'].replace(r'^\s*$', np.nan, regex=True)

# Menghitung jumlah string kosong yang terdeteksi sebelum di-drop
print(f"String kosong ditemukan di Train      : {df_train['text_clean'].isnull().sum()} baris")
print(f"String kosong ditemukan di Validation : {df_valid['text_clean'].isnull().sum()} baris")
print(f"String kosong ditemukan di Test       : {df_test['text_clean'].isnull().sum()} baris")

# Menghapus baris yang text_clean nya bernilai NaN/None
df_train = df_train.dropna(subset=['text_clean']).reset_index(drop=True)
df_valid = df_valid.dropna(subset=['text_clean']).reset_index(drop=True)
df_test = df_test.dropna(subset=['text_clean']).reset_index(drop=True)

print(f"\n--> [SUCCESS] Audit selesai. Dimensi data akhir operasional:")
print(f"Data Train      : {df_train.shape[0]} baris")
print(f"Data Validation : {df_valid.shape[0]} baris")
print(f"Data Test       : {df_test.shape[0]} baris")

=== 1. Memulai Proses Preprocessing Massal... ===
--> [SUCCESS] Pembuatan kolom 'text_clean' pada seluruh dataset selesai!

=== 2. Audit & Penanganan String Kosong ===
String kosong ditemukan di Train      : 0 baris
String kosong ditemukan di Validation : 0 baris
String kosong ditemukan di Test       : 0 baris

--> [SUCCESS] Audit selesai. Dimensi data akhir operasional:
Data Train      : 10933 baris
Data Validation : 1260 baris
Data Test       : 500 baris


### INSIGHT & ANALISIS HASIL TAHAPAN 4: PIPELINE MASSAL & AUDIT DATA

Berdasarkan hasil eksekusi pemrosesan massal dan evaluasi filter pada sel di atas, diperoleh tiga kesimpulan penting sebagai berikut:

1. **Skalabilitas dan Efisiensi Pipeline Teruji**
   Fungsi gabungan `total_clean_pipeline` mampu mengeksekusi transformasi teks secara *end-to-end* (Cleansing $\rightarrow$ Tokenisasi $\rightarrow$ Normalisasi $\rightarrow$ Detokenisasi) pada belasan ribu baris data secara simultan dengan sangat cepat. Kolom baru bernama `text_clean` sukses terbentuk di ketiga subset data tanpa menimbulkan galat memori (*runtime error*).

2. **Integritas Kandungan Informasi Data Riil (0 Baris String Kosong)**
   Hasil audit menunjukkan angka **0 (nol) baris** string kosong yang terdeteksi di seluruh subset data (*Train, Validation, Test*). 
   * **Analisis Semantik:** Temuan ini membuktikan bahwa tidak ada satu pun dokumen ulasan di dalam korpus *SmSA* Alvian yang isinya murni berupa *noise* digital (seperti hanya berisi angka `"100%"` atau emoji saja). 
   * Setiap ulasan digital netizen terbukti mengandung minimal satu kata alfabet bermakna konotatif/denotatif, sehingga tidak ada informasi berharga yang terbuang sia-sia akibat proses penyaringan regex di Langkah 2.

3. **Kombinasi Final Dimensi Operasional Dataset**
   Selesainya tahapan audit ini secara resmi mengunci volume data operasional yang akan kita alirkan ke tahap perancangan model:
   * **Data Train:** `10.933 baris` (Bersih dari duplikat & string kosong).
   * **Data Validation:** `1.260 baris` (Utuh untuk keperluan *Proxy Test Set*).
   * **Data Test:** `500 baris` (Utuh untuk keperluan *True Inference/Submission*).
     
> **Catatan Teknis Implementasi:** Proses konversi string kosong memanfaatkan pola ekspresi reguler `r'^\s*$'` untuk mendeteksi *whitespace* tersembunyi. Penggunaan objek standar `np.nan` dikombinasikan dengan metode `.dropna()` menjamin arsitektur kode kita aman dari *FutureWarning* Pandas serta memastikan data benar-benar steril sebelum masuk ke tahap ekstraksi fitur TF-IDF.

## LANGKAH 5: ANALISIS EKSPLORATIF DISTRIBUSI KELAS SENTIMEN (TARGET LABEL EDA)

Sebelum melangkah ke tahap pemodelan atau ekstraksi fitur pada Notebook berikutnya, kita wajib melakukan *Exploratory Data Analysis* (EDA) khusus pada kolom target label (`sentiment`). Pembedahan statistik deskriptif ini bertujuan untuk mengetahui sebaran proporsi data latih secara riil, mendeteksi potensi ketidakseimbangan kelas (*class imbalance*), serta merumuskan strategi pemilihan metrik evaluasi performa model yang adil dan objektif.

In [5]:
# ==============================================================================
# TAHAPAN 5: ANALISIS EKSPLORATIF TARGET SENTIMEN (EDA TARGET)
# ==============================================================================

# 1. Menghitung jumlah baris absolut per kelas sentimen
jumlah_sentimen = df_train['sentiment'].value_counts()

# 2. Menghitung proporsi persentase (%) per kelas sentimen
persentase_sentimen = df_train['sentiment'].value_counts(normalize=True) * 100

# 3. Menggabungkan hasil ke dalam satu DataFrame ringkas untuk laporan
tabel_eda_target = pd.DataFrame({
    'Jumlah Baris': jumlah_sentimen,
    'Persentase (%)': persentase_sentimen
})

print("=== DISTRIBUSI TARGET LABEL KATEGORI (DATA TRAIN) ===")
print(tabel_eda_target)

=== DISTRIBUSI TARGET LABEL KATEGORI (DATA TRAIN) ===
           Jumlah Baris  Persentase (%)
sentiment                              
positive           6383       58.382878
negative           3412       31.208269
neutral            1138       10.408854


### INSIGHT & ANALISIS HASIL TAHAPAN 5: TARGET LABEL EDA

Berdasarkan tabel distribusi sentimen operasional di atas, kita berhasil mengungkap karakteristik fundamental dari dataset korpus *SmSA* Alvian/IndoNLU sebagai berikut:

1. **Kondisi *Imbalanced Dataset* yang Nyata**
   Dataset ini terbukti mengalami masalah **ketidakseimbangan kelas (*class imbalance*)** yang cukup kontras. Struktur label target tidak tersebar merata secara ideal (33.3% per kelas), melainkan terpolarisasi ke dalam tiga tingkatan volume data:
   * **Kelas Mayoritas (*Majority Class*):** Dipegang oleh sentimen `positive` dengan porsi dominan sebesar **58.38%** (6.383 baris). Komentar berupa pujian, kepuasan, atau dukungan mendominasi lebih dari separuh total data latih.
   * **Kelas Moderat:** Dipegang oleh sentimen `negative` dengan porsi menengah sebesar **31.21%** (3.412 baris) yang berisi teks kritikan atau kekecewaan.
   * **Kelas Minoritas (*Minority Class*):** Dipegang oleh sentimen `neutral` yang bersifat sangat langka, hanya menyisakan porsi sebesar **10.41%** (1.138 baris) berupa kalimat fakta atau berita datar.

2. **Formulasi Strategi Evaluasi Pemodelan (Konsekuensi Krusial)**
   * **Larangan Penggunaan Akurasi Tunggal:** Mengingat kondisi data yang timpang ini, kita **dilarang keras** menggunakan metrik *Accuracy* (Akurasi) sebagai tolok ukur tunggal keberhasilan model di Notebook 2 nanti. Jika model mengalami kelumpuhan dan menebak asal ke arah kelas `positive` saja, akurasi sistem otomatis sudah menyentuh angka ~58%, padahal model gagal total mengenali sentimen `neutral` dan `negative`.
   * **Solusi Arsitektur Portofolio:** Untuk mengantisipasi bias mayoritas tersebut, kita wajib mengadopsi metrik **Macro F1-Score** sebagai kompas evaluasi utama, didampingi oleh evaluasi *Precision* dan *Recall* per kelas. Metrik *Macro Average* akan merata-ratakan performa ketiga kelas secara berimbang tanpa memedulikan dominasi jumlah baris individu. Dengan demikian, kemampuan model dalam mendeteksi kelas minoritas (`neutral`) akan tetap dinilai dan diuji secara adil dan objektif pada tahapan kompetisi model.

## LANGKAH 6: MENYIMPAN DATASET HASIL PREPROCESSING (EXPORT DATASET)

Langkah terakhir dari rangkaian Notebook 1 ini adalah mengekspor dan menyimpan secara permanen ketiga dataset yang telah melalui proses audit serta pembersihan teks (`df_train`, `df_valid`, dan `df_test`). 

Data disimpan ke dalam direktori khusus data siap pakai (`../data/processed/`) menggunakan format `.tsv` agar konsisten dengan ekstensi dokumen mentah aslinya. Proses ekspor ini mengunci hasil kerja keras kita agar siap dialirkan langsung menuju Notebook 2 untuk tahap pemodelan (*feature engineering & modeling*).

In [6]:
# ==============================================================================
# TAHAPAN 6: EXPORT PROCESSED DATASET (.TSV FISIK)
# ==============================================================================

print("=== Menyimpan Dataset Hasil Preprocessing ===")
# 1. Menentukan nama folder tujuan ekspor data
output_dir = "../data/processed/"

# 2. Membuat folder secara otomatis jika folder tersebut belum tersedia
os.makedirs(output_dir, exist_ok=True)

# 3. Mengekspor dataframe menjadi file fisik .tsv tanpa menyertakan indeks bawaan pandas
df_train.to_csv(os.path.join(output_dir, "train_clean.tsv"), sep='\t', index=False)
df_valid.to_csv(os.path.join(output_dir, "valid_clean.tsv"), sep='\t', index=False)
df_test.to_csv(os.path.join(output_dir, "test_clean.tsv"), sep='\t', index=False)

print(f"--> [SUCCESS] File 'train_clean.tsv' ({df_train.shape[0]} baris) berhasil disimpan.")
print(f"--> [SUCCESS] File 'valid_clean.tsv' ({df_valid.shape[0]} baris) berhasil disimpan.")
print(f"--> [SUCCESS] File 'test_clean.tsv'  ({df_test.shape[0]} baris) berhasil disimpan.")
print(f"\n==================================================\n--> SEMUA FILE BERSIH BERHASIL DIAMANKAN DI: {output_dir}\n==================================================")

=== Menyimpan Dataset Hasil Preprocessing ===
--> [SUCCESS] File 'train_clean.tsv' (10933 baris) berhasil disimpan.
--> [SUCCESS] File 'valid_clean.tsv' (1260 baris) berhasil disimpan.
--> [SUCCESS] File 'test_clean.tsv'  (500 baris) berhasil disimpan.

--> SEMUA FILE BERSIH BERHASIL DIAMANKAN DI: ../data/processed/


### INSIGHT & ANALISIS HASIL TAHAPAN 6: EXPORT DATASET

Proses penyimpanan data fisik ini menandai selesainya seluruh rangkaian pengerjaan pra-pemrosesan awal dengan beberapa poin evaluasi teknis sebagai berikut:

1. **Arsitektur Penyimpanan yang Idempotent**
   Dengan memanfaatkan fungsi `os.makedirs(output_dir, exist_ok=True)`, sistem memastikan bahwa direktori target `../data/processed/` dijamin tersedia di mesin lokal. Penggunaan parameter `exist_ok=True` mencegah timbulnya galat sistem (*FileExistsError*) seandainya folder tersebut sudah dibuat pada eksekusi script sebelumnya, sehingga runtime kode tetap berjalan aman dan stabil.

2. **Mitigasi *Data Leakage* (Kebocoran Data) Secara Ketat**
   Pengeksporan ketiga berkas dilakukan secara terpisah dengan menjaga segregasi baris aslinya semenjak dari korpus awal Kaggle Alvian. Dengan mengunci `train_clean.tsv`, `valid_clean.tsv`, dan `test_clean.tsv` pada partisi fisiknya masing-masing, kita menjamin tidak akan ada pencampuran kosakata (*vocabulary leakage*) dari data uji ke data latih saat vectorizer (TF-IDF) dibangun di Notebook 2.

3. **Efisiensi Rekayasa Data (*Data Engineering Workflow*)**
   Penyimpanan tanpa menyertakan indeks bawaan pandas (`index=False`) memastikan file `.tsv` yang terbentuk murni hanya berisi data operasional yang padat. Dengan terkuncinya kolom `text_clean` dan `sentiment`, kita dapat menutup kernel Notebook 1 ini dengan aman. Pada Notebook 2 nanti, kita cukup memanggil berkas olahan ini secara instan tanpa perlu membuang waktu komputasi untuk mengulang proses *preprocessing* dari awal.